Feature Engineering (32 Features - 3 Groups)

1. Setup & Load Data

In [1]:
import os

# Fix path tuyệt đối để tránh lỗi relative path trên Windows
BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROCESSED_DIR = os.path.join(BASE_DIR, 'Data', 'Processed')
FEATURES_DIR  = os.path.join(BASE_DIR, 'Data', 'Features')
HOLDOUT_DIR   = os.path.join(BASE_DIR, 'Data', 'Holdout')

os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(HOLDOUT_DIR,  exist_ok=True)

print(f"BASE_DIR      : {BASE_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"FEATURES_DIR  : {FEATURES_DIR}")
print(f"HOLDOUT_DIR   : {HOLDOUT_DIR}")
print(f"Dirs exist    : {all(os.path.exists(d) for d in [PROCESSED_DIR, FEATURES_DIR, HOLDOUT_DIR])}")

BASE_DIR      : d:\Study Assets\Final Project\PhishingGuard
PROCESSED_DIR : d:\Study Assets\Final Project\PhishingGuard\Data\Processed
FEATURES_DIR  : d:\Study Assets\Final Project\PhishingGuard\Data\Features
HOLDOUT_DIR   : d:\Study Assets\Final Project\PhishingGuard\Data\Holdout
Dirs exist    : True


In [2]:
import pandas as pd
import numpy as np
import re
import math
from urllib.parse import urlparse, parse_qs
from sklearn.model_selection import train_test_split

# ── Load 3 datasets ──
df1 = pd.read_csv(os.path.join(PROCESSED_DIR, 'dataset1_clean.csv'))
df2 = pd.read_csv(os.path.join(PROCESSED_DIR, 'dataset2_clean.csv'))
dfm = pd.read_csv(os.path.join(PROCESSED_DIR, 'merged_dataset.csv'))

print("── Loaded datasets ──")
print(f"  Dataset 1 : {df1.shape} | columns: {list(df1.columns)}")
print(f"  Dataset 2 : {df2.shape} | columns: {list(df2.columns)}")
print(f"  Merged    : {dfm.shape} | columns: {list(dfm.columns)}")

── Loaded datasets ──
  Dataset 1 : (637179, 2) | columns: ['URL', 'Label']
  Dataset 2 : (129776, 2) | columns: ['URL', 'Label']
  Merged    : (766882, 3) | columns: ['URL', 'Label', 'source']


2. Stratified Split 80/20

In [3]:
datasets = [
    ('dataset1', df1, 'Label',
     os.path.join(FEATURES_DIR, 'dataset1_features.csv'),
     os.path.join(HOLDOUT_DIR,  'dataset1_holdout.csv')),
    ('dataset2', df2, 'Label',
     os.path.join(FEATURES_DIR, 'dataset2_features.csv'),
     os.path.join(HOLDOUT_DIR,  'dataset2_holdout.csv')),
    ('merged',   dfm, 'Label',
     os.path.join(FEATURES_DIR, 'merged_features.csv'),
     os.path.join(HOLDOUT_DIR,  'merged_holdout.csv')),
]

splits = {}

print("── Stratified Split 80/20 ──")
for name, df, label_col, feat_path, hold_path in datasets:
    train, holdout = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df[label_col]
    )
    train   = train.reset_index(drop=True)
    holdout = holdout.reset_index(drop=True)

    # Lưu holdout ngay — không đụng vào nữa
    holdout.to_csv(hold_path, index=False)

    splits[name] = {
        'train'     : train,
        'feat_path' : feat_path,
        'label_col' : label_col,
    }

    vc_train   = train[label_col].value_counts().to_dict()
    vc_holdout = holdout[label_col].value_counts().to_dict()

    print(f"\n  [{name}]")
    print(f"  Train   : {len(train):,} rows | labels: {vc_train}")
    print(f"  Holdout : {len(holdout):,} rows | labels: {vc_holdout}")
    print(f"  Holdout saved → {os.path.basename(hold_path)}")

print("\n Holdout đã lưu — không dùng đến khi test cuối")

── Stratified Split 80/20 ──

  [dataset1]
  Train   : 509,743 rows | labels: {'benign': 339307, 'defacement': 76246, 'phishing': 75274, 'malware': 18916}
  Holdout : 127,436 rows | labels: {'benign': 84827, 'defacement': 19062, 'phishing': 18818, 'malware': 4729}
  Holdout saved → dataset1_holdout.csv

  [dataset2]
  Train   : 103,820 rows | labels: {0: 59977, 1: 43843}
  Holdout : 25,956 rows | labels: {0: 14995, 1: 10961}
  Holdout saved → dataset2_holdout.csv

  [merged]
  Train   : 613,505 rows | labels: {0: 399278, 1: 214227}
  Holdout : 153,377 rows | labels: {0: 99820, 1: 53557}
  Holdout saved → merged_holdout.csv

 Holdout đã lưu — không dùng đến khi test cuối


3. Feature Function Definitions

In [4]:
# ── Constants ──
SUSPICIOUS_WORDS = [
    'login', 'signin', 'verify', 'verification', 'secure', 'security',
    'account', 'update', 'confirm', 'banking', 'payment', 'password',
    'credential', 'authenticate', 'wallet', 'alert', 'suspended', 'unusual'
]
SUSPICIOUS_TLDS = {
    'tk', 'ml', 'ga', 'cf', 'gq', 'xyz', 'top', 'club', 'work',
    'click', 'link', 'online', 'site', 'live', 'download', 'stream'
}
URL_SHORTENERS = {
    'bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly', 'rebrand.ly',
    'buff.ly', 'is.gd', 'short.link', 'cutt.ly', 'tiny.cc'
}
VOWELS     = set('aeiouAEIOU')
CONSONANTS = set('bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ')

# ── Helper ──
def _parse(url):
    try:    return urlparse(str(url))
    except: return urlparse('')

def _domain(url):
    return _parse(url).netloc.split(':')[0].lower()

def _shannon(s):
    if not s: return 0.0
    freq = {}
    for c in s: freq[c] = freq.get(c, 0) + 1
    n = len(s)
    return -sum((f/n) * math.log2(f/n) for f in freq.values())

#   LEXICAL FEATURES (19 features)

def f_url_length(u):          return len(u)
def f_domain_length(u):       return len(_domain(u))
def f_path_length(u):         return len(_parse(u).path)
def f_num_dots(u):            return u.count('.')
def f_num_hyphens(u):         return u.count('-')
def f_num_underscores(u):     return u.count('_')
def f_num_slashes(u):         return u.count('/')
def f_num_at(u):              return u.count('@')
def f_num_question_marks(u):  return u.count('?')
def f_num_equals(u):          return u.count('=')
def f_num_ampersands(u):      return u.count('&')
def f_num_percent(u):         return u.count('%')
def f_num_digits(u):          return sum(c.isdigit() for c in u)

def f_digit_ratio(u):
    return f_num_digits(u) / len(u) if len(u) > 0 else 0.0

def f_letter_ratio(u):
    return sum(c.isalpha() for c in u) / len(u) if len(u) > 0 else 0.0

def f_url_entropy(u):         return round(_shannon(u), 4)
def f_domain_entropy(u):      return round(_shannon(_domain(u)), 4)

def f_num_subdomains(u):
    parts = _domain(u).split('.')
    return max(0, len(parts) - 2)

def f_has_ip_address(u):
    d = _domain(u)
    m = re.match(r'^(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.(\d{1,3})$', d)
    return int(bool(m) and all(0 <= int(g) <= 255 for g in m.groups())) if m else 0

# STRUCTURAL FEATURES (8 features)

def f_has_https(u):
    return int(str(u).lower().startswith('https://'))

def f_is_suspicious_tld(u):
    parts = _domain(u).split('.')
    return int(parts[-1] in SUSPICIOUS_TLDS) if parts else 0

def f_has_port(u):
    return int(':' in _parse(u).netloc)

def f_path_depth(u):
    parts = [p for p in _parse(u).path.split('/') if p]
    return len(parts)

def f_num_query_params(u):
    q = _parse(u).query
    return len(parse_qs(q)) if q else 0

def f_has_fragment(u):
    return int(bool(_parse(u).fragment))

def f_is_shortened_url(u):
    d = _domain(u).lstrip('www.')
    return int(d in URL_SHORTENERS)

def f_has_suspicious_word(u):
    return int(any(w in str(u).lower() for w in SUSPICIOUS_WORDS))

# STATISTICAL FEATURES (5 features)

def f_consonant_ratio(u):
    letters = [c for c in u if c.isalpha()]
    return sum(1 for c in letters if c in CONSONANTS) / len(letters) if letters else 0.0

def f_vowel_ratio(u):
    letters = [c for c in u if c.isalpha()]
    return sum(1 for c in letters if c in VOWELS) / len(letters) if letters else 0.0

def f_longest_token_length(u):
    tokens = re.split(r'[/.\-_?=&%@]', u)
    return max((len(t) for t in tokens if t), default=0)

def f_avg_token_length(u):
    tokens = [t for t in re.split(r'[/.\-_?=&%@]', u) if t]
    return round(sum(len(t) for t in tokens) / len(tokens), 4) if tokens else 0.0

def f_special_char_ratio(u):
    special = sum(1 for c in u if not c.isalnum())
    return round(special / len(u), 4) if len(u) > 0 else 0.0


print(" 32 feature functions defined")
print(f"\n  Lexical    (19): url_length, domain_length, path_length, num_dots,")
print(f"                   num_hyphens, num_underscores, num_slashes, num_at,")
print(f"                   num_question_marks, num_equals, num_ampersands, num_percent,")
print(f"                   num_digits, digit_ratio, letter_ratio,")
print(f"                   url_entropy, domain_entropy, num_subdomains, has_ip_address")
print(f"\n  Structural  (8): has_https, is_suspicious_tld, has_port, path_depth,")
print(f"                   num_query_params, has_fragment, is_shortened_url, has_suspicious_word")
print(f"\n  Statistical (5): consonant_ratio, vowel_ratio,")
print(f"                   longest_token_length, avg_token_length, special_char_ratio")

 32 feature functions defined

  Lexical    (19): url_length, domain_length, path_length, num_dots,
                   num_hyphens, num_underscores, num_slashes, num_at,
                   num_question_marks, num_equals, num_ampersands, num_percent,
                   num_digits, digit_ratio, letter_ratio,
                   url_entropy, domain_entropy, num_subdomains, has_ip_address

  Structural  (8): has_https, is_suspicious_tld, has_port, path_depth,
                   num_query_params, has_fragment, is_shortened_url, has_suspicious_word

  Statistical (5): consonant_ratio, vowel_ratio,
                   longest_token_length, avg_token_length, special_char_ratio


4.  Extract & Export

In [5]:
FEATURE_FUNCS = [
    # Lexical
    ('url_length',          f_url_length),
    ('domain_length',       f_domain_length),
    ('path_length',         f_path_length),
    ('num_dots',            f_num_dots),
    ('num_hyphens',         f_num_hyphens),
    ('num_underscores',     f_num_underscores),
    ('num_slashes',         f_num_slashes),
    ('num_at',              f_num_at),
    ('num_question_marks',  f_num_question_marks),
    ('num_equals',          f_num_equals),
    ('num_ampersands',      f_num_ampersands),
    ('num_percent',         f_num_percent),
    ('num_digits',          f_num_digits),
    ('digit_ratio',         f_digit_ratio),
    ('letter_ratio',        f_letter_ratio),
    ('url_entropy',         f_url_entropy),
    ('domain_entropy',      f_domain_entropy),
    ('num_subdomains',      f_num_subdomains),
    ('has_ip_address',      f_has_ip_address),
    # Structural
    ('has_https',           f_has_https),
    ('is_suspicious_tld',   f_is_suspicious_tld),
    ('has_port',            f_has_port),
    ('path_depth',          f_path_depth),
    ('num_query_params',    f_num_query_params),
    ('has_fragment',        f_has_fragment),
    ('is_shortened_url',    f_is_shortened_url),
    ('has_suspicious_word', f_has_suspicious_word),
    # Statistical
    ('consonant_ratio',     f_consonant_ratio),
    ('vowel_ratio',         f_vowel_ratio),
    ('longest_token_length',f_longest_token_length),
    ('avg_token_length',    f_avg_token_length),
    ('special_char_ratio',  f_special_char_ratio),
]

def extract_features(df, label_col, dataset_name):
    urls = df['URL'].astype(str)
    print(f"\n── Extracting: {dataset_name} ({len(df):,} rows) ──")

    feat_df = pd.DataFrame({'URL': urls, label_col: df[label_col].values})

    for feat_name, func in FEATURE_FUNCS:
        feat_df[feat_name] = urls.apply(func)
        print(f"  ✓ {feat_name}", end='\r')

    print(f"Done — shape: {feat_df.shape}          ")
    return feat_df

print("── Feature Extraction ──")
for name, info in splits.items():
    train     = info['train']
    feat_path = info['feat_path']
    label_col = info['label_col']

    feat_df = extract_features(train, label_col, name)
    feat_df.to_csv(feat_path, index=False)
    print(f"  Exported → {os.path.basename(feat_path)}")

print("\nFeature extraction hoàn tất cho cả 3 datasets")

── Feature Extraction ──

── Extracting: dataset1 (509,743 rows) ──
Done — shape: (509743, 34)          
  Exported → dataset1_features.csv

── Extracting: dataset2 (103,820 rows) ──
Done — shape: (103820, 34)          
  Exported → dataset2_features.csv

── Extracting: merged (613,505 rows) ──
Done — shape: (613505, 34)          
  Exported → merged_features.csv

Feature extraction hoàn tất cho cả 3 datasets


5. Validation

In [6]:
print("── Validation ──\n")

feat_files = [
    ('Dataset 1', os.path.join(FEATURES_DIR, 'dataset1_features.csv')),
    ('Dataset 2', os.path.join(FEATURES_DIR, 'dataset2_features.csv')),
    ('Merged',    os.path.join(FEATURES_DIR, 'merged_features.csv')),
]

for name, path in feat_files:
    df = pd.read_csv(path)
    nulls   = df.isnull().sum().sum()
    infs    = np.isinf(df.select_dtypes(include=np.number)).sum().sum()

    print(f"[{name}]")
    print(f"  Shape       : {df.shape}")
    print(f"  Null values : {nulls}")
    print(f"  Inf values  : {infs}")

    if nulls > 0:
        print(f"Null breakdown:")
        print(df.isnull().sum()[df.isnull().sum() > 0].to_string())
    if infs > 0:
        print(f" Inf breakdown:")
        num_df = df.select_dtypes(include=np.number)
        print(np.isinf(num_df).sum()[np.isinf(num_df).sum() > 0].to_string())
    if nulls == 0 and infs == 0:
        print(f"Clean — không có null hoặc inf")

    print(f"\n  Basic stats (5 features mẫu):")
    sample_cols = ['url_length', 'url_entropy', 'digit_ratio',
                   'num_subdomains', 'special_char_ratio']
    print(df[sample_cols].describe().round(3).to_string())
    print()

── Validation ──

[Dataset 1]
  Shape       : (509743, 34)
  Null values : 0
  Inf values  : 0
Clean — không có null hoặc inf

  Basic stats (5 features mẫu):
       url_length  url_entropy  digit_ratio  num_subdomains  special_char_ratio
count  509743.000   509743.000   509743.000      509743.000          509743.000
mean       64.817        4.302        0.065           0.599               0.185
std        43.961        0.351        0.093           1.071               0.039
min         8.000        2.130        0.000           0.000               0.011
25%        38.000        4.067        0.000           0.000               0.162
50%        52.000        4.286        0.028           0.000               0.182
75%        81.000        4.526        0.095           1.000               0.205
max      2182.000        7.493        0.763          33.000               0.667

[Dataset 2]
  Shape       : (103820, 34)
  Null values : 0
  Inf values  : 0
Clean — không có null hoặc inf

  Basic sta